# Additional spectral-band baseline: two Random Forest classifiers

This independent notebook reads the common sample table, the previously saved partition, and the Traditional_RF_RF model parameters. It trains the additional baseline without retraining or overwriting the previous three groups.

Fill in `COMMON_PATH`, `PREVIOUS_RESULTS`, and `OUTPUT_PARENT` before running. No local paths are supplied. Use the same Python environment as the three-group experiment. tqdm is optional; a built-in progress display is available.

Inputs are 40 quarterly spectral-band variables (four quarters, ten bands). They retain cloud masking and quarterly median compositing. No vegetation indices, topography, or AEF features are included. The 119,736 common samples and saved training/test assignments are reused without resampling or filtering.

RF parameters are read from `Traditional_RF_RF/model_parameters.json` in the previous result directory. Outputs cover forest/non-forest discrimination, five forest types on reference-labelled forest test samples, and final six-class cascade predictions. Metrics include OA, Kappa, Macro-F1, Weighted-F1, and binary forest F1.

Four-group comparison tables and an efficiency table are generated. Combined confusion-matrix panels are generated when all previous matrix CSVs are available. Previous models, metrics, and source files remain unchanged.

The previous three groups retain their recorded timings; this group is measured in a separate session. These are supplementary efficiency observations, not a single-session controlled speed benchmark. Thread limits, batch size, warm-up, and repeat counts follow the previous settings.

Results are written to a new timestamped directory under the configured output parent. Complete additional-group outputs are stored in its `RawBands_RF_RF` subdirectory.


In [ ]:
from pathlib import Path
from datetime import datetime
import os,sys,json,time,hashlib,platform,shutil
try:
    import numpy as np
    import pandas as pd
    import sklearn
    import matplotlib.pyplot as plt
    import joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score,cohen_kappa_score,f1_score,confusion_matrix,classification_report
    from threadpoolctl import threadpool_limits
except ImportError as exc:
    raise RuntimeError('Use the same Python environment as the three-group experiment. Install missing dependencies in a separate cell: %pip install numpy pandas scikit-learn matplotlib joblib threadpoolctl') from exc

COMMON_PATH = ""
PREVIOUS_RESULTS = ""
OUTPUT_PARENT = ""
CLASS5=['EBF','DBF','ENF','DNF','MF']
CLASS6=CLASS5+['Others']
META=['source_id','source_dataset','label_year','feature_year','lc_code','label_binary','final_class','lon','lat']
BANDS=['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
RAW_FEATURES=['Q{}_{}'.format(q,b) for q in range(1,5) for b in BANDS]
FOREST_MAP={50:'EBF',60:'DBF',70:'ENF',80:'DNF',90:'MF'}
NEW_GROUP='RawBands_RF_RF'
OLD_GROUPS=['AEF_main_ET_KNN','AEF_control_RF_RF','Traditional_RF_RF']
STAGES=['step1_binary','step2_true_forest','final_six_class']
# These three runtime settings are overwritten by the original run_config.
N_JOBS=4
PREDICT_BATCH_SIZE=2048
TIMING_REPEATS=3


def require_path(value, setting):
    if value is None or not str(value).strip():
        raise ValueError("Set " + setting + " before running this notebook.")
    return Path(value)


def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def export_csv(df,path):
    df.to_csv(path,index=False,encoding='utf-8-sig')

def evaluate(y_true,y_pred,labels):
    return {'OA':accuracy_score(y_true,y_pred),
            'Kappa':cohen_kappa_score(y_true,y_pred),
            'Macro_F1':f1_score(y_true,y_pred,labels=labels,average='macro',zero_division=0),
            'Weighted_F1':f1_score(y_true,y_pred,labels=labels,average='weighted',zero_division=0)}

def save_evaluation(y_true,y_pred,labels,names,stage,out,group):
    cm=confusion_matrix(y_true,y_pred,labels=labels)
    cm_df=pd.DataFrame(cm,index=names,columns=names)
    cm_df.index.name='True_class'
    cm_df.to_csv(out/(stage+'_confusion_counts.csv'),encoding='utf-8-sig')
    norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros(cm.shape,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    norm_df=pd.DataFrame(norm,index=names,columns=names);norm_df.index.name='True_class'
    norm_df.to_csv(out/(stage+'_confusion_row_normalized.csv'),encoding='utf-8-sig')
    report=classification_report(y_true,y_pred,labels=labels,target_names=names,output_dict=True,zero_division=0)
    export_csv(pd.DataFrame([{'class':name,**report[name]} for name in names]),out/(stage+'_per_class_metrics.csv'))
    fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
    for ax,values,title in zip(axes,[cm,norm],['Counts','Recall by true class']):
        im=ax.imshow(values,cmap='Blues',vmin=0,vmax=None if title=='Counts' else 1)
        ax.set_xticks(range(len(names)));ax.set_yticks(range(len(names)))
        ax.set_xticklabels(names,rotation=45,ha='right');ax.set_yticklabels(names)
        ax.set_xlabel('Predicted class');ax.set_ylabel('True class');ax.set_title(title)
        for i in range(len(names)):
            for j in range(len(names)):
                text=str(int(values[i,j])) if title=='Counts' else '{:.1%}'.format(values[i,j])
                threshold=(values.max()/2) if title=='Counts' else .5
                ax.text(j,i,text,ha='center',va='center',color='white' if values[i,j]>threshold else 'black',fontsize=9)
        fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
    fig.suptitle(group+' / '+stage)
    fig.savefig(out/(stage+'_confusion.png'),dpi=300,bbox_inches='tight');plt.close(fig)
    return evaluate(y_true,y_pred,labels)

def predict_batched(model,scaler,X,progress=False,desc='Prediction'):
    if len(X)==0:return np.array([],dtype=object)
    ranges=range(0,len(X),PREDICT_BATCH_SIZE)
    if progress:ranges=tqdm(ranges,desc=desc,unit='batch',leave=False)
    chunks=[]
    for start in ranges:
        part=X[start:start+PREDICT_BATCH_SIZE]
        if scaler is not None:part=scaler.transform(part)
        chunks.append(model.predict(part))
    return np.concatenate(chunks)

def cascade_predict(model1,model2,scaler2,X):
    binary=predict_batched(model1,None,X)
    pred=np.full(len(X),'Others',dtype=object)
    routed=binary==1
    if routed.any():pred[routed]=predict_batched(model2,scaler2,X[routed])
    return binary,pred

def time_prediction(action,repeats):
    # One full untimed warm-up, followed by repeated full-test-set measurements.
    reference=action();times=[]
    for _ in range(repeats):
        t=time.perf_counter();result=action();times.append(time.perf_counter()-t)
        first=reference if isinstance(reference,tuple) else (reference,)
        last=result if isinstance(result,tuple) else (result,)
        if not all(np.array_equal(a,b) for a,b in zip(first,last)):
            raise RuntimeError('Repeated predictions are inconsistent. Stopping timing aggregation.')
    return reference,float(np.median(times)),times

# Use tqdm when installed; otherwise show a built-in text progress bar.
try:
    from tqdm import tqdm
except ImportError:
    import time
    class tqdm:
        def __init__(self,iterable=None,total=None,desc='',unit='',unit_scale=False,leave=True,**kwargs):
            self.iterable=iterable;self.total=total if total is not None else len(iterable)
            self.desc=desc;self.unit=unit;self.n=0;self.last=0;self.suffix=''
        def __enter__(self):
            self.show(True);return self
        def __exit__(self,*args):
            self.show(True);print('',flush=True)
        def __iter__(self):
            self.show(True)
            for item in self.iterable:
                yield item
                self.update(1)
            self.show(True);print('',flush=True)
        def update(self,n=1):
            self.n+=n;self.show()
        def set_postfix_str(self,text):
            self.suffix=text;self.show(True)
        def show(self,force=False):
            now=time.monotonic()
            if not force and now-self.last<0.3:return
            ratio=min(1,self.n/max(self.total,1));filled=int(ratio*20)
            print('\r{} [{}{}] {:6.1%} {}/{} {} {}'.format(
                self.desc,'█'*filled,'·'*(20-filled),ratio,self.n,self.total,self.unit,self.suffix),end='',flush=True)
            self.last=now





In [ ]:

def check_metadata(original,other,description):
    for c in META:
        if c not in other:raise ValueError(description+' is missing field: '+c)
        if c in ['label_year','feature_year','lc_code','label_binary','lon','lat']:
            equal=np.isclose(pd.to_numeric(original[c]).to_numpy(float),pd.to_numeric(other[c]).to_numpy(float),rtol=0,atol=1e-8)
        else:equal=original[c].astype(str).to_numpy()==other[c].astype(str).to_numpy()
        if not np.all(equal):raise ValueError(description+' does not match the common sample table: '+c)


def saved_split(df,previous,out):
    split=pd.read_csv(previous/'shared_sample_split.csv',encoding='utf-8-sig',dtype={'source_id':str})
    if len(split)!=len(df) or split.source_id.isna().any() or split.source_id.duplicated().any() or set(split.source_id)!=set(df.source_id):
        raise ValueError('Saved split sample identifiers do not match. A new split will not be generated.')
    # Restore shared CSV order by source_id, then align split identities.
    aligned=split.set_index('source_id',drop=False).loc[df.source_id].reset_index(drop=True)
    check_metadata(df,aligned,'Previous split')
    if set(aligned['split'])!={'train','test'}:raise ValueError('The previous split contains invalid subset labels.')
    train=np.flatnonzero(aligned['split'].eq('train').to_numpy())
    test=np.flatnonzero(aligned['split'].eq('test').to_numpy())
    locations=np.array([format(x,'.6f')+'_'+format(y,'.6f') for x,y in zip(df.lon,df.lat)])
    if set(locations[train])&set(locations[test]):raise ValueError('The previous split places coincident locations in both training and testing. Stopping for review.')
    if set(df.iloc[train].final_class)!=set(CLASS6) or set(df.iloc[test].final_class)!=set(CLASS6):raise ValueError('The training or test set is missing a target class.')
    # Reproduce the ORIGINAL file-order digest used by the earlier notebook.
    digest=hashlib.sha256('\n'.join(split.source_id+'|'+split['split']).encode('utf-8')).hexdigest()
    shutil.copy2(previous/'shared_sample_split.csv',out/'shared_sample_split.csv')
    counts=aligned.groupby(['split','label_year','final_class']).size().rename('n').reset_index()
    export_csv(counts,out/'shared_split_counts.csv')
    return train,test,digest


def load_previous_metrics(previous,digest,n_test,n_forest):
    tables={}
    for stage in STAGES:
        p=previous/('comparison_'+stage+'.csv')
        table=pd.read_csv(p,encoding='utf-8-sig')
        needed={'Group','Stage','N_test','OA','Kappa','Macro_F1','Weighted_F1','Split_SHA256'}
        if not needed.issubset(table.columns):raise ValueError('Missing columns in the previous metric table: '+str(p))
        if len(table)!=3 or set(table.Group)!=set(OLD_GROUPS) or table.Group.duplicated().any():raise ValueError('The previous metric table does not contain the expected three groups: '+str(p))
        expected_n=n_forest if stage=='step2_true_forest' else n_test
        if not table.Stage.eq(stage).all() or not table.Split_SHA256.eq(digest).all() or not table.N_test.eq(expected_n).all():raise ValueError('Inconsistent split identifiers or sample counts in the previous metric table: '+str(p))
        tables[stage]=table
    efficiency=pd.read_csv(previous/'efficiency_comparison.csv',encoding='utf-8-sig')
    if len(efficiency)!=3 or set(efficiency.Group)!=set(OLD_GROUPS) or not efficiency.Split_SHA256.eq(digest).all():raise ValueError('The previous efficiency table has inconsistent groups or split identifiers.')
    if not efficiency.N_test.eq(n_test).all():raise ValueError('The previous efficiency table has inconsistent test sample counts.')
    return tables,efficiency


def optional_four_group_panels(previous,out,new_dir):
    # Existing matrices are read, not reconstructed from rounded screenshots.
    groups=OLD_GROUPS+[NEW_GROUP]
    missing=[]
    for stage in STAGES:
        for group in OLD_GROUPS:
            p=previous/group/(stage+'_confusion_row_normalized.csv')
            if not p.is_file():missing.append(str(p))
    if missing:
        (out/'missing_old_matrix_files.txt').write_text('\n'.join(missing),encoding='utf-8')
        print('Previous confusion-matrix CSVs are incomplete. Additional-group figures and four-group metric tables are saved; only the combined panels are skipped.',flush=True)
        return
    for stage in STAGES:
        names=['Nonforest','Forest'] if stage=='step1_binary' else CLASS5 if stage=='step2_true_forest' else CLASS6
        fig,axes=plt.subplots(1,4,figsize=(22,5),constrained_layout=True)
        for ax,group in zip(axes,groups):
            root=new_dir if group==NEW_GROUP else previous/group
            cm=pd.read_csv(root/(stage+'_confusion_row_normalized.csv'),index_col=0)
            if list(cm.columns)!=names or list(cm.index)!=names:raise ValueError('Confusion-matrix class order does not match: '+str(root))
            values=cm.to_numpy(float)
            im=ax.imshow(values,cmap='Blues',vmin=0,vmax=1)
            ax.set_xticks(range(len(names)));ax.set_yticks(range(len(names)))
            ax.set_xticklabels(names,rotation=45,ha='right');ax.set_yticklabels(names)
            ax.set_title(group);ax.set_xlabel('Predicted class');ax.set_ylabel('True class')
            for i in range(len(names)):
                for j in range(len(names)):
                    ax.text(j,i,'{:.1%}'.format(values[i,j]),ha='center',va='center',color='white' if values[i,j]>.5 else 'black',fontsize=8)
        fig.colorbar(im,ax=axes.ravel().tolist(),fraction=.015,pad=.02)
        fig.suptitle(stage+' / shared test samples / row-normalized')
        fig.savefig(out/('four_groups_'+stage+'_confusion.png'),dpi=300,bbox_inches='tight');plt.close(fig)


def run_raw_bands(input_path=None,previous_results=None,output_parent=None,expected_count=119736):
    global N_JOBS,PREDICT_BATCH_SIZE,TIMING_REPEATS
    input_path = require_path(input_path if input_path is not None else COMMON_PATH, 'COMMON_PATH')
    previous = require_path(previous_results if previous_results is not None else PREVIOUS_RESULTS, 'PREVIOUS_RESULTS')
    parent = require_path(output_parent if output_parent is not None else OUTPUT_PARENT, 'OUTPUT_PARENT')
    if not input_path.is_file():raise FileNotFoundError('Common sample table not found: '+str(input_path))
    required=[previous/'run_config.json',previous/'shared_sample_split.csv',previous/'Traditional_RF_RF'/'model_parameters.json',previous/'efficiency_comparison.csv']
    required += [previous/('comparison_'+stage+'.csv') for stage in STAGES]
    for p in required:
        if not p.is_file():raise FileNotFoundError('Missing previous result file: '+str(p))
    config=json.loads((previous/'run_config.json').read_text(encoding='utf-8'))
    rf=json.loads((previous/'Traditional_RF_RF'/'model_parameters.json').read_text(encoding='utf-8'))
    params1=rf['step1'];params2=rf['step2']
    N_JOBS=int(config['n_jobs']);PREDICT_BATCH_SIZE=int(config['prediction_batch_size']);TIMING_REPEATS=int(config['timing_repeats'])
    if N_JOBS<1 or PREDICT_BATCH_SIZE<1 or TIMING_REPEATS<1:raise ValueError('The previous run configuration contains invalid settings.')
    if params1.get('n_jobs')!=N_JOBS or params2.get('n_jobs')!=N_JOBS:raise ValueError('Previous RF thread parameters do not match the run configuration.')
    out=parent/('raw_bands_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'));out.mkdir(parents=True,exist_ok=False)
    gd=out/NEW_GROUP;gd.mkdir()
    notes=[]
    def log(text):
        print(text,flush=True);notes.append(text);(out/'summary.txt').write_text('\n'.join(notes),encoding='utf-8')
    log('Training the additional group only: 40 quarterly spectral-band features, RF to RF. Previous groups are not retrained or overwritten.')
    log('Previous split directory: '+str(previous));log('Output: '+str(out))
    with tqdm(total=9,desc='Additional-group progress',unit='stage') as bar, sklearn.config_context(working_memory=256),threadpool_limits(limits=N_JOBS):
        sha=sha256_file(input_path)
        if sha!=config.get('input_sha256'):raise ValueError('The common sample file SHA256 differs from the previous input. Stopping without generating a new split.')
        df=pd.read_csv(input_path,encoding='utf-8-sig',dtype={'source_id':str})
        if not set(META+RAW_FEATURES).issubset(df.columns):raise ValueError('Required fields are missing.')
        if expected_count is not None and len(df)!=expected_count:raise ValueError('Expected {} common samples; found {}.'.format(expected_count,len(df)))
        if df.source_id.isna().any() or df.source_id.duplicated().any():raise ValueError('source_id contains missing or duplicate values.')
        for c in RAW_FEATURES+['lon','lat','lc_code','label_year','feature_year','label_binary']:df[c]=pd.to_numeric(df[c],errors='raise')
        if not np.isfinite(df[RAW_FEATURES].to_numpy(float)).all() or (df[RAW_FEATURES]==-9999).any().any():raise ValueError('Spectral bands contain missing or invalid values. Rows will not be dropped automatically.')
        if not df.lc_code.map(FOREST_MAP).fillna('Others').eq(df.final_class).all() or not df.label_binary.eq(df.final_class.ne('Others').astype(int)).all():raise ValueError('Class labels are inconsistent.')
        df=df.sort_values('source_id',kind='mergesort').reset_index(drop=True)
        bar.update(1);bar.set_postfix_str('Validating and reusing saved split')
        train,test,digest=saved_split(df,previous,out)
        if digest!=config.get('split_sha256'):raise ValueError('The previous split digest does not match. Stopping.')
        binary=df.label_binary.to_numpy(int);classes=df.final_class.to_numpy()
        tf=train[binary[train]==1];vf=test[binary[test]==1]
        old_tables,old_eff=load_previous_metrics(previous,digest,len(test),len(vf))
        if len(train)!=config['n_train'] or len(test)!=config['n_test'] or len(tf)!=config['n_train_forest'] or len(vf)!=config['n_test_forest']:raise ValueError('Training or test counts do not match the previous configuration.')
        log('Reused split: training {}; testing {}; forest training {}; reference-labelled forest testing {}.'.format(len(train),len(test),len(tf),len(vf)))
        log('Previous RF parameters loaded: {} trees at both stages.'.format(params1['n_estimators']))
        bar.update(1)
        X=df[RAW_FEATURES].to_numpy(dtype=np.float64)
        Xt=np.ascontiguousarray(X[train]);Xv=np.ascontiguousarray(X[test]);Xtf=np.ascontiguousarray(X[tf]);Xvf=np.ascontiguousarray(X[vf])
        model1=RandomForestClassifier(**params1);model2=RandomForestClassifier(**params2)
        bar.set_postfix_str('Stage 1 RF training');log('Training the additional stage-1 forest/non-forest RF...')
        t=time.perf_counter();model1.fit(Xt,binary[train]);fit1=time.perf_counter()-t;bar.update(1)
        bar.set_postfix_str('Stage 2 RF training');log('Training the additional stage-2 five-class forest-type RF...')
        t=time.perf_counter();model2.fit(Xtf,classes[tf]);fit2=time.perf_counter()-t;bar.update(1)
        bar.set_postfix_str('Prediction and timing');log('Predicting and timing: median of {} full predictions after warm-up.'.format(TIMING_REPEATS))
        p1=predict_batched(model1,None,Xv,True,'Stage 1 prediction');p2=predict_batched(model2,None,Xvf,True,'Stage 2 prediction')
        with tqdm(total=3,desc='Efficiency measurement',unit='task',leave=False) as tb:
            p1r,s1,r1=time_prediction(lambda:predict_batched(model1,None,Xv),TIMING_REPEATS);tb.update(1)
            p2r,s2,r2=time_prediction(lambda:predict_batched(model2,None,Xvf),TIMING_REPEATS);tb.update(1)
            (cp1,final),sf,rf_times=time_prediction(lambda:cascade_predict(model1,model2,None,Xv),TIMING_REPEATS);tb.update(1)
        if not np.array_equal(p1,p1r) or not np.array_equal(p1,cp1) or not np.array_equal(p2,p2r):raise RuntimeError('Predictions are inconsistent.')
        bar.update(1);bar.set_postfix_str('Saving metrics and confusion matrices')
        rows=[]
        for stage,yt,yp,labels,names in [
            ('step1_binary',binary[test],p1,[0,1],['Nonforest','Forest']),
            ('step2_true_forest',classes[vf],p2,CLASS5,CLASS5),
            ('final_six_class',classes[test],final,CLASS6,CLASS6)]:
            metric=save_evaluation(yt,yp,labels,names,stage,gd,NEW_GROUP)
            metric.update(Group=NEW_GROUP,Stage=stage,Feature_set='Quarterly_raw_bands_40',Step1_model='RF',Step2_model='RF',N_test=len(yt),Split_SHA256=digest)
            metric['Forest_F1']=f1_score(yt,yp,pos_label=1,zero_division=0) if stage=='step1_binary' else np.nan
            rows.append(metric)
        new_metrics=pd.DataFrame(rows);export_csv(new_metrics,gd/'raw_bands_metrics.csv')
        pred=df.iloc[test][META].reset_index(drop=True).copy();pred['step1_pred_binary']=p1;pred['step2_pred_on_true_forest']=''
        pred.loc[binary[test]==1,'step2_pred_on_true_forest']=p2;pred['cascade_pred_class']=final
        export_csv(pred,gd/'test_predictions.csv')
        for name,model in [('step1',model1),('step2',model2)]:
            export_csv(pd.DataFrame({'feature':RAW_FEATURES,'importance':model.feature_importances_}).sort_values('importance',ascending=False),gd/(name+'_feature_importance.csv'))
        (gd/'model_parameters.json').write_text(json.dumps({'step1':model1.get_params(),'step2':model2.get_params(),'step2_scaler':None},ensure_ascii=False,indent=2),encoding='utf-8')
        bar.update(1);bar.set_postfix_str('Saving additional models')
        joblib.dump({'group':NEW_GROUP,'step1':model1,'step2':model2,'step2_scaler':None,'features':RAW_FEATURES,'class5':CLASS5,'split_sha256':digest},gd/'models.joblib',compress=3)
        runtime={'Group':NEW_GROUP,'Step1_fit_seconds':fit1,'Step2_fit_seconds':fit2,'Total_fit_seconds':fit1+fit2,
            'Step1_predict_median_seconds':s1,'Step2_true_forest_predict_median_seconds':s2,'Cascade_predict_median_seconds':sf,
            'Cascade_samples_per_second':len(test)/max(sf,1e-12),'N_test':len(test),'N_test_true_forest':len(vf),'N_predicted_forest':int((p1==1).sum()),
            'Timing_repeats':TIMING_REPEATS,'N_jobs':N_JOBS,'Prediction_batch_size':PREDICT_BATCH_SIZE,'Split_SHA256':digest,
            'Saved_models_MiB':(gd/'models.joblib').stat().st_size/(1024**2)}
        export_csv(pd.DataFrame([runtime]),gd/'raw_bands_efficiency.csv')
        (gd/'timing_repetitions.json').write_text(json.dumps({'step1_predict_seconds':r1,'step2_true_forest_predict_seconds':r2,'cascade_predict_seconds':rf_times},indent=2),encoding='utf-8')
        bar.update(1);bar.set_postfix_str('Appending four-group comparison tables')
        all_tables=[]
        for stage in STAGES:
            four=pd.concat([old_tables[stage],new_metrics.loc[new_metrics.Stage==stage]],ignore_index=True)
            export_csv(four,out/('comparison_'+stage+'.csv'));all_tables.append(four)
        combined=pd.concat(all_tables,ignore_index=True);export_csv(combined,out/'four_groups_all_stages_metrics.csv')
        old_eff=old_eff.copy();old_eff['Timing_session']='previous_three_group_run'
        new_eff=pd.DataFrame([dict(runtime,Timing_session='current_raw_bands_run')])
        efficiency=pd.concat([old_eff,new_eff],ignore_index=True)
        efficiency['Timing_note']='Two sessions; same recorded thread/batch settings, not a single-session controlled speed benchmark.'
        export_csv(efficiency,out/'efficiency_comparison.csv')
        bar.update(1);bar.set_postfix_str('Four-group confusion-matrix panels')
        optional_four_group_panels(previous,out,gd)
        record={'input_path':str(input_path),'input_sha256':sha,'previous_results':str(previous),'split_sha256':digest,
            'n_samples':len(df),'n_train':len(train),'n_test':len(test),'n_train_forest':len(tf),'n_test_forest':len(vf),
            'features':RAW_FEATURES,'trained_models_this_run':['RawBands_RF_RF_step1','RawBands_RF_RF_step2'],
            'old_groups_retrained':False,'old_metrics_source':'Previous comparison_*.csv, copied without modification of numeric results.',
            'software_current':{'python':sys.version,'sklearn':sklearn.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()},
            'software_previous':{'python':config.get('python'),'sklearn':config.get('sklearn'),'numpy':config.get('numpy'),'pandas':config.get('pandas')},
            'efficiency_scope':'Original three groups reuse previous times. New group measured now. No GEE/extraction/download/AEF pretraining included.',
            'n_jobs':N_JOBS,'prediction_batch_size':PREDICT_BATCH_SIZE,'timing_repeats':TIMING_REPEATS,
            'model_parameters_source':str(previous/'Traditional_RF_RF'/'model_parameters.json')}
        (out/'run_config.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
        readme=['This run trains only the two RF models using 40 spectral-band features. The previous three groups are not retrained.',
            'Loads the previous shared_sample_split.csv without resampling or filtering. Common samples, labels, input SHA256, and the split digest are validated.',
            'RF parameters are loaded from Traditional_RF_RF/model_parameters.json. max_features remains sqrt; the number of candidate features therefore changes with input dimensionality.',
            'Spectral bands are cloud-masked quarterly median composites, not unprocessed imagery. No vegetation indices, topography, or AEF embeddings are used.',
            'Four overall metrics are reported for binary classification, five-class classification on reference-labelled forest, and final six-class cascade prediction with error propagation. Binary classification also reports forest F1.',
            'Confusion-matrix rows are reference classes and columns are predictions. Per-class precision is UA and recall is PA. Combined panels use exact CSV values rather than values reconstructed from screenshots.',
            'The previous three-group models and full predictions remain in their original result directory. Additional-group outputs are stored in RawBands_RF_RF.',
            'Four-group accuracy tables share the same samples. Previous timing measurements are reused for the first three groups, while the additional group is measured in a separate session. Efficiency is supplementary, not a single-session controlled speed benchmark.',
            'Model size is the compressed file size in MiB at compression level 3, not peak memory usage.',
            'Original data and results are unchanged. Reruns create new timestamped directories.']
        (out/'README.txt').write_text('\n'.join(readme),encoding='utf-8')
        log('Additional group completed:\n'+new_metrics[['Stage','N_test','OA','Kappa','Macro_F1','Weighted_F1','Forest_F1']].to_string(index=False))
        log('Four-group tables saved. Training {:.2f} s; median cascade prediction {:.3f} s.'.format(fit1+fit2,sf))
        log('Outputs include comparison_*.csv, efficiency_comparison.csv, and four_groups_*_confusion.png.')
        bar.update(1);bar.set_postfix_str('Complete')
    return out,new_metrics,combined,efficiency


In [ ]:
RESULT_DIR, RAW_BANDS_METRICS, FOUR_GROUPS_METRICS, FOUR_GROUPS_EFFICIENCY = run_raw_bands()
